# Project Work 2 · La catena di montaggio di un modello

**Modulo 1.18 · Lezioni 13 e 14 (4 + 4 ore)**

## Dove vogliamo arrivare

Un magazzino di ricambi industriali ha 7.771 articoli. Per ognuno sappiamo quanti pezzi sono usciti
ogni mese, da aprile 2025 ad agosto 2026 (17 mesi). I numeri sono veri, gli articoli sono stati resi anonimi.

Il vostro gruppo deve costruire, pezzo per pezzo, un sistema che:

1. **prevede** quanti pezzi usciranno il mese prossimo, per ogni articolo di una famiglia di prodotti;
2. **decide** quanta scorta tenere (la scorta di sicurezza) e quando fare l'ordine (il punto di riordino).

Venerdì ricevete un mese che oggi non avete: **settembre 2026**. Nessuno di voi lo ha mai visto, nemmeno il modello.
Sarà la prova del nove: il sistema che avete costruito oggi funziona su un mese nuovo? Poi confrontate la vostra
decisione con quella del sistema che l'azienda usa davvero.

## Perché si chiama "catena di montaggio"

Nelle lezioni avete fatto spesso domande come: *come faccio a scegliere il modello? devo riallenarlo ogni volta?
come scelgo gli iperparametri?* La risposta a tutte è la stessa: **non si decide a occhio, si segue un procedimento.**
Un procedimento fatto di passi sempre uguali, come una catena di montaggio: i dati entrano da una parte,
passano per una serie di stazioni, e dall'altra parte esce una decisione.

Il notebook è diviso in 8 **blocchi**. Ogni blocco è una stazione della catena e risponde a una domanda.

| Blocco | Domanda | Quando |
|---|---|---|
| 1 | Con quali dati lavoro? | oggi |
| 2 | Che cosa do in pasto al modello? | oggi |
| 3 | Quando il test è onesto? | oggi |
| 4 | Il modello batte una regola semplice? | oggi |
| 5 | Come scelgo il modello? | oggi |
| 6 | Come scelgo gli iperparametri? | oggi |
| 7 | Devo riallenarlo ogni volta? | venerdì |
| 8 | E poi, che cosa ci faccio? | venerdì |

## Come lavoriamo

- Formate gruppi di **3 o 4 persone**, come preferite.
- Ruoli, uno per persona (si possono scambiare a metà pomeriggio):
  - **Tastiera**: scrive ed esegue il codice.
  - **Mappa**: cerca nella mappa del corso (glossario, "Come scelgo il modello", schede delle lezioni) e nella documentazione di scikit-learn.
    Non è un ruolo di riserva: nei blocchi 2, 5 e 6 senza la Mappa non si va avanti.
  - **Diario**: compila la scheda del gruppo in fondo al notebook, man mano, non alla fine.
  - **Voce**: presenta venerdì (5 minuti).
- Ogni blocco ha tre parti: una **spiegazione** (leggetela tutta, davvero), una cella di **codice da completare**
  e una cella di **Controllo**.
- Nel codice ci sono due tipi di lavoro:
  - dove trovate `FILL_ME` manca un pezzo preciso: il commento accanto dice **che cosa** deve fare, non come. Il come lo trovate voi.
  - dove trovate `# --- scrivete voi ---` manca una parte intera, e le scelte sono vostre. Le istruzioni sono nel testo sopra la cella.
- La cella di **Controllo** stampa `OK` se il blocco è a posto, altrimenti un messaggio che dice che cosa non va.
  Non andate avanti senza l'`OK`.
- **Il test si usa una volta sola.** Le scelte (colonne, modello, iperparametri) si fanno sul train con la cross-validation.
  Chi guarda il test più volte non sta barando con il docente: sta barando con se stesso, perché i numeri che ottiene
  non dicono più niente sul futuro.
- Eseguite le celle **in ordine**, dall'alto in basso. Se qualcosa si rompe: `Kernel → Restart & Run All`.
- Obiettivo di oggi: **blocco 6 finito**, con il modello scelto e "sigillato". Venerdì i blocchi 7 e 8 e la presentazione.

## Che cosa consegnate venerdì

1. Il notebook eseguito, con tutti i controlli a `OK`.
2. La **scheda del gruppo** in fondo, compilata.
3. Una presentazione di 5 minuti: la famiglia scelta, le colonne che avete inventato, il modello scelto e perché,
   com'è andata su settembre, quanto vi discostate dal sistema dell'azienda.

## La gara tra i gruppi

Ogni gruppo lavora su una famiglia diversa, e le famiglie non sono tutte uguali: in una si sbaglia di 3 pezzi al mese,
in un'altra di 20. Quindi la gara non è su chi sbaglia meno, ma su **chi migliora di più rispetto alla regola semplice**
(la baseline del blocco 4). Alla fine del blocco 6 una cella stampa il vostro miglioramento in percentuale:
scrivetelo alla lavagna accanto al nome del gruppo. Venerdì si rifà la classifica su settembre.

## Preparazione

Eseguite questa cella così com'è. Carica le librerie e i dati.

Il file `warehouse_history.csv` deve stare nella stessa cartella del notebook. Se la cella dà errore
`FileNotFoundError`, il file è da un'altra parte.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV
from sklearn.base import clone

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

# carico il file con lo storico
data = pd.read_csv("warehouse_history.csv")

# i mesi sono le colonne che iniziano con un anno (2025-04, 2025-05, ...)
MONTHS = [column for column in data.columns if column[:4].isdigit()]

print("Articoli:", len(data))
print("Mesi:", MONTHS[0], "->", MONTHS[-1], "(", len(MONTHS), "mesi )")
data.head()

### Che cosa c'è nel file

Ogni riga è un articolo. Le colonne:

| Colonna | Significato |
|---|---|
| `item_id` | numero dell'articolo (anonimo, non dice niente dell'articolo vero) |
| `macro_family` | famiglia di prodotto, per esempio "Tenute e guarnizioni" |
| `product_class` | classe più fine dentro la famiglia |
| `unit_cost` | costo di un pezzo, in euro |
| `lead_time_days` | giorni che passano tra l'ordine al fornitore e l'arrivo della merce in magazzino |
| `abc_class` | A = articoli che pesano molto sul fatturato, B medio, C poco |
| `xyz_class` | X = domanda regolare, Y media, Z irregolare |
| `service_level` | quante volte su 100 l'azienda vuole avere il pezzo disponibile (0,85 = 85 volte su 100) |
| `k` | il coefficiente che corrisponde al livello di servizio (serve nel blocco 8) |
| `2025-04` ... `2026-08` | pezzi usciti dal magazzino in quel mese |

Le colonne dei mesi sono lo **storico**: è l'unica cosa che il modello può imparare. Tutto il resto descrive l'articolo.

Eseguite la cella qui sotto: mostra le famiglie e quanti articoli ha ognuna. Ogni gruppo sceglie **una famiglia diversa**
(mettetevi d'accordo tra gruppi).

In [ ]:
# conto quanti articoli ha ogni famiglia
items_per_family = data["macro_family"].value_counts()
items_per_family

## Blocco 1 · Con quali dati lavoro?

**La domanda.** Prima di qualsiasi modello bisogna decidere **su che cosa** lo si addestra. Sembra ovvio, ma è
la decisione che più spesso rovina un progetto: un modello addestrato su dati che non c'entrano, o su articoli che non
hanno niente da insegnare, dà risultati che sembrano buoni e non valgono niente.

**L'idea.** Un articolo che è uscito 2 volte in 17 mesi non si può prevedere: non c'è un comportamento da imparare,
solo rumore. Teniamo quindi solo gli articoli della vostra famiglia che hanno **almeno 12 pezzi usciti in tutto lo storico**.
Il numero 12 è una scelta, non una legge: in un'azienda vera lo deciderebbe il responsabile del magazzino con voi.

**Che cosa dovete fare.**

1. Scrivete il nome esatto della vostra famiglia in `FAMILY`, copiandolo dalla tabella qui sopra (maiuscole, accenti e spazi contano).
2. Completate `load_family`: manca la condizione che tiene solo gli articoli con almeno `min_total` pezzi in tutto lo storico.
   `total_sold` è già calcolato: è la somma dei mesi, riga per riga.

Scegliete una famiglia che dopo il filtro abbia **almeno 100 articoli** (il controllo ve lo dice). Le famiglie grandi sono
Tenute e guarnizioni, Valvole del compressore, Organi del cilindro, Bulloneria e serraggio, Altro.
Se volete una famiglia piccola, unitene due: al posto di `data["macro_family"] == family` usate
`data["macro_family"].isin(["Filtrazione", "Pompe e macchine"])`.

**Come sapete che funziona.** Il controllo verifica che ci siano articoli, che siano almeno 100 e che nessuno abbia
meno di `MIN_TOTAL` pezzi.

In [ ]:
FAMILY = "Tenute e guarnizioni"     # <- la nostra famiglia
MIN_TOTAL = 12


def load_family(data, family, min_total):
    # tengo solo le righe della famiglia scelta
    is_family = data["macro_family"] == family
    family_data = data[is_family].copy()

    # sommo i pezzi usciti in tutti i mesi, articolo per articolo (una somma per riga)
    total_sold = family_data[MONTHS].sum(axis=1)

    # tengo solo gli articoli con abbastanza movimento (almeno min_total pezzi)
    is_active = total_sold >= min_total

    family_data = family_data[is_active].copy()
    family_data = family_data.reset_index(drop=True)
    return family_data


items = load_family(data, FAMILY, MIN_TOTAL)
print("Articoli della famiglia", FAMILY, "con almeno", MIN_TOTAL, "pezzi:", len(items))
items.head()

In [ ]:
# --- Controllo blocco 1 ---
assert len(items) > 0, "Nessun articolo: controllate il nome della famiglia (maiuscole, accenti, spazi)"
assert len(items) >= 100, f"Solo {len(items)} articoli: scegliete una famiglia più grande o unitene due"
assert items[MONTHS].sum(axis=1).min() >= MIN_TOTAL, "C'è ancora qualche articolo con meno di MIN_TOTAL pezzi: controllate la condizione"
print("OK · blocco 1 ·", len(items), "articoli")

**Guardate i dati prima di andare avanti.** È la cosa che nessuno fa e che tutti dovrebbero fare.
La cella mostra lo storico di 6 articoli presi a caso. Eseguitela più volte cambiando `random_state` per vederne altri.

Domande per il Diario: ci sono articoli regolari (escono più o meno ogni mese)? Articoli a ondate (niente per mesi, poi un blocco)?
Articoli con un solo mese grande e il resto zero? Quale di questi tre tipi vi aspettate che sia più difficile da prevedere, e perché?
Scrivetelo nella scheda del gruppo: vi servirà per inventare le colonne del blocco 2.

In [ ]:
# disegno lo storico di 6 articoli presi a caso
sample_items = items.sample(6, random_state=1)

figure, axes = plt.subplots(2, 3, figsize=(14, 6))
axes = axes.flatten()
for position in range(6):
    row = sample_items.iloc[position]
    axes[position].bar(range(len(MONTHS)), row[MONTHS].values)
    axes[position].set_title(f"articolo {row['item_id']} · classe {row['abc_class']}{row['xyz_class']} · lead time {row['lead_time_days']} gg")
    axes[position].set_xticks([0, 6, 12, 16])
    axes[position].set_xticklabels([MONTHS[0], MONTHS[6], MONTHS[12], MONTHS[16]], fontsize=8)
plt.tight_layout()
plt.show()

## Blocco 2 · Che cosa do in pasto al modello?

**La domanda.** Un modello di scikit-learn non legge uno storico: legge una tabella `X` di numeri (le colonne che conosciamo)
e una colonna `y` (quello che vogliamo prevedere). Trasformare lo storico in questa tabella è il lavoro più importante
di tutto il progetto, e nessuna libreria lo fa per voi. Nella lezione 5 lo avete già fatto una volta, quando avete costruito
la colonna "distanza" per il deposito: era una colonna inventata da voi, e ha cambiato il risultato.

**L'idea.** Ogni riga di `X` è **un articolo in un mese**. Chiamiamo quel mese il **mese bersaglio**:

- `y` (il bersaglio) = i pezzi usciti nel mese bersaglio;
- `X` = quello che sapevamo **prima** del mese bersaglio.

Lo stesso articolo compare più volte, una per ogni mese bersaglio: così da 17 mesi di storico ricaviamo molti esempi
(per 800 articoli e 11 mesi bersaglio, quasi 9.000 righe). Il modello impara dalla domanda
"dato quello che vedevo prima, quanto è uscito dopo?", ripetuta migliaia di volte.

**La regola d'oro.** Nelle colonne di `X` ci va **solo quello che si sapeva prima del mese bersaglio**. Mai il mese bersaglio
stesso, mai un mese dopo. Se per sbaglio ci finisce, il modello sembra perfetto sul test e poi nella realtà non funziona:
si chiama **fuga di dati** (cercatela nel glossario della mappa). È l'errore più comune e più difficile da vedere.

**Che cosa dovete fare.**

La funzione `make_examples` riceve gli articoli e un mese bersaglio, e costruisce le righe di quel mese.
Vi diamo già tre colonne e il bersaglio:

- `lag_1`, `lag_2`, `lag_3`: i pezzi usciti 1, 2 e 3 mesi prima del bersaglio;
- `target`: i pezzi usciti nel mese bersaglio.

Dovete **inventare e aggiungere almeno 3 colonne in più**. Qualche idea, ma le scelte sono vostre:

- la media, la deviazione standard o il massimo degli ultimi 6 mesi (`previous_6_values` è già pronto: sono le 6 colonne prima del bersaglio);
- quanti degli ultimi 6 mesi sono stati a zero (per gli articoli "a ondate");
- lo stesso mese dell'anno prima, se esiste (per la stagionalità; attenzione: per i primi mesi bersaglio non c'è);
- informazioni fisse dell'articolo: la classe ABC trasformata in numero (A=3, B=2, C=1), il lead time, il costo unitario.

Per ogni colonna scrivete nella scheda **perché** dovrebbe aiutare. Non aggiungete colonne a caso: ogni colonna in più
è una cosa in più che il modello deve capire, e con pochi dati le colonne inutili peggiorano il risultato.

Le colonne devono essere **numeri** (niente testo come "A" o "Tenute") e **senza valori mancanti**.

**Come sapete che funziona.** Il controllo ricostruisce gli esempi di agosto 2026 dopo aver **azzerato** agosto 2026 nei dati:
se le colonne di `X` cambiano, vuol dire che avete usato il mese bersaglio, cioè avete una fuga di dati. Controlla anche che
ci siano almeno 6 colonne, tutte numeriche, senza valori mancanti.

In [ ]:
ABC_TO_NUMBER = {"A": 3, "B": 2, "C": 1}


def make_examples(items, target_month):
    # posizione del mese bersaglio nella lista dei mesi
    target_position = MONTHS.index(target_month)

    # le 6 colonne dei mesi prima del bersaglio (già pronte per i vostri calcoli)
    previous_6 = MONTHS[target_position - 6:target_position]
    previous_6_values = items[previous_6]

    examples = pd.DataFrame()
    examples["item_id"] = items["item_id"]
    examples["target_month"] = target_month

    # i 3 mesi subito prima del bersaglio
    examples["lag_1"] = items[MONTHS[target_position - 1]]
    examples["lag_2"] = items[MONTHS[target_position - 2]]
    examples["lag_3"] = items[MONTHS[target_position - 3]]

    # --- scrivete voi: almeno 3 colonne in più ---
    # ricordate: solo informazioni note PRIMA del mese bersaglio


    # il bersaglio: i pezzi usciti nel mese bersaglio
    examples["target"] = items[target_month]

    return examples


# costruisco gli esempi per tutti i mesi che hanno almeno 6 mesi prima
example_tables = []
for month in MONTHS[6:]:
    examples_of_month = make_examples(items, month)
    example_tables.append(examples_of_month)
examples = pd.concat(example_tables, ignore_index=True)

# le colonne di X sono tutte quelle che avete costruito, tranne identificativi e bersaglio
FEATURES = [column for column in examples.columns if column not in ["item_id", "target_month", "target"]]

print("Esempi:", len(examples), "=", len(items), "articoli x", len(MONTHS[6:]), "mesi bersaglio")
print("Colonne di X:", FEATURES)
examples.head()

In [ ]:
# --- Controllo blocco 2 ---
assert len(FEATURES) >= 6, f"Avete {len(FEATURES)} colonne: ne servono almeno 6 (i 3 lag più almeno 3 vostre)"
assert examples[FEATURES].isna().sum().sum() == 0, "Ci sono valori mancanti in X: controllate le colonne che avete aggiunto"
for column in FEATURES:
    assert pd.api.types.is_numeric_dtype(examples[column]), f"La colonna {column} non è numerica: trasformatela in numeri"

# prova della fuga di dati: azzero agosto 2026 e ricostruisco gli esempi di agosto 2026
items_without_august = items.copy()
items_without_august["2026-08"] = 0
examples_original = make_examples(items, "2026-08").reset_index(drop=True)
examples_altered = make_examples(items_without_august, "2026-08").reset_index(drop=True)
same_features = examples_original[FEATURES].equals(examples_altered[FEATURES])
assert same_features, "FUGA DI DATI: una colonna di X cambia se cambio il mese bersaglio. In X ci va solo quello che si sapeva prima"
print("OK · blocco 2 ·", len(examples), "esempi ·", len(FEATURES), "colonne in X:", FEATURES)

## Blocco 3 · Quando il test è onesto?

**La domanda.** Nella lezione 3 avete imparato a tenere da parte un pezzo dei dati per il test, con `train_test_split`.
Quella funzione **mescola le righe a caso**. Qui non va bene, e capire perché è il punto di questo blocco.

**L'idea.** I nostri esempi hanno una data. Se mescoliamo a caso, nel train finiscono righe di agosto 2026 e nel test righe
di luglio 2026: il modello impara dal futuro e viene provato sul passato. Sul test sembrerà bravo, ma nella realtà
non avrà mai il futuro a disposizione. Per i dati nel tempo il taglio si fa **nel tempo**:

- **train** = esempi con mese bersaglio da `2025-10` a `2026-06` (9 mesi);
- **test** = esempi con mese bersaglio `2026-07` e `2026-08` (gli ultimi 2 mesi che avete).

Il test imita quello che succede davvero: addestro su tutto quello che so fino a giugno, poi vedo come va a luglio e agosto.
E venerdì arriva settembre, che è il test più onesto di tutti: non esiste ancora nel vostro file.

**Che cosa dovete fare.** Completate `split_by_time`: manca la condizione che è vera per le righe dei mesi di test.
`examples["target_month"]` contiene il mese come testo (`"2026-07"`), `test_months` è una lista di testi.

**Come sapete che funziona.** Il controllo verifica che nel test ci siano solo i mesi di `TEST_MONTHS`, che nel train
non ci sia nessun mese del test o dopo, e che il test abbia esattamente 2 righe per articolo.

In [ ]:
TEST_MONTHS = ["2026-07", "2026-08"]


def split_by_time(examples, test_months):
    # vero per le righe il cui mese bersaglio è uno dei mesi di test
    is_test = FILL_ME     # scrivete: la condizione "vero" per le righe con il mese bersaglio nella lista test_months

    train_examples = examples[~is_test].copy()
    test_examples = examples[is_test].copy()

    X_train = train_examples[FEATURES]
    y_train = train_examples["target"]
    X_test = test_examples[FEATURES]
    y_test = test_examples["target"]

    return X_train, y_train, X_test, y_test, train_examples, test_examples


X_train, y_train, X_test, y_test, train_examples, test_examples = split_by_time(examples, TEST_MONTHS)

print("Train:", len(X_train), "esempi · mesi", sorted(train_examples["target_month"].unique()))
print("Test: ", len(X_test), "esempi · mesi", sorted(test_examples["target_month"].unique()))

In [ ]:
# --- Controllo blocco 3 ---
assert set(test_examples["target_month"]) == set(TEST_MONTHS), "Nel test ci devono essere solo i mesi di TEST_MONTHS"
assert train_examples["target_month"].max() < min(TEST_MONTHS), "Nel train c'è un mese del test o del futuro"
assert len(X_test) == len(TEST_MONTHS) * len(items), "Il test deve avere una riga per articolo per ogni mese di test"
print("OK · blocco 3 · train", len(X_train), "· test", len(X_test))

## Blocco 4 · Il modello batte una regola semplice?

**La domanda.** "Il modello ha una MAE di 9,5." È tanto? È poco? Da solo il numero non dice niente.
Serve un confronto, e il confronto giusto è con la regola più semplice che una persona userebbe senza machine learning.
Si chiama **baseline**. Se il modello non batte la baseline, il modello non serve: si usa la regola.

**L'idea.** La nostra baseline è quella della lezione 2: *il mese prossimo uscirà la media degli ultimi 3 mesi.*
È quello che farebbe un magazziniere con un foglio di calcolo.

La misura dell'errore è la **MAE** (errore assoluto medio, lezione 2): per ogni riga prendo la differenza tra previsione e realtà,
tolgo il segno, e faccio la media. Si legge in **pezzi**: "sbagliamo in media di 9,5 pezzi per articolo al mese".
È più facile da spiegare della RMSE, e qui ci interessa che il responsabile del magazzino capisca.

**Che cosa dovete fare.** Completate `baseline_predict`: la previsione è la media di `lag_1`, `lag_2` e `lag_3`, riga per riga.

**Come sapete che funziona.** Il controllo rifà il calcolo a mano sulla prima riga del test.

In [ ]:
def baseline_predict(X):
    # la previsione è la media dei 3 mesi precedenti, riga per riga
    three_months = X[["lag_1", "lag_2", "lag_3"]]
    prediction = FILL_ME     # scrivete: la media delle tre colonne, calcolata per ogni riga
    return prediction


baseline_prediction = baseline_predict(X_test)
baseline_mae = mean_absolute_error(y_test, baseline_prediction)

print(f"Baseline (media 3 mesi) · MAE sul test = {baseline_mae:.2f} pezzi")

In [ ]:
# --- Controllo blocco 4 ---
first_row = X_test.iloc[0]
expected = (first_row["lag_1"] + first_row["lag_2"] + first_row["lag_3"]) / 3
assert abs(baseline_prediction.iloc[0] - expected) < 1e-9, "La baseline deve essere la media di lag_1, lag_2, lag_3 per ogni riga"
print(f"OK · blocco 4 · il numero da battere è MAE = {baseline_mae:.2f} pezzi")

## Blocco 5 · Come scelgo il modello?

**La domanda.** È la domanda che avete fatto più spesso. La risposta è: **non si sceglie, si misura.**
Il procedimento è sempre lo stesso, per qualsiasi problema:

1. tutti i modelli candidati vedono **gli stessi dati** di train;
2. tutti vengono misurati **con la stessa misura** (qui la MAE) **sugli stessi dati** di test;
3. si tiene quello con l'errore più basso, **a patto che batta la baseline**. Se nessuno la batte, si tiene la baseline;
4. a parità di errore si tiene il più semplice: più facile da spiegare, più veloce, meno cose che si rompono.

Qui stiamo prevedendo **un numero** (pezzi al mese), quindi è un problema di **regressione**, non di classificazione:
in scikit-learn servono i modelli con "Regressor" nel nome. Quasi tutti i modelli che avete visto esistono in entrambe le versioni.

**Che cosa dovete fare.**

1. Completate `train_model` (addestra il modello sui dati di train) ed `evaluate_model` (calcola la MAE sul test).
2. Riempite il dizionario `candidate_models` con **almeno 4 modelli, presi da almeno 3 lezioni diverse** del corso.
   La Mappa cerca nella pagina "Come scelgo il modello" e nelle schede delle lezioni quali modelli avete incontrato e come si chiamano
   in scikit-learn nella versione per la regressione. Le librerie sono già importate nella cella di preparazione: guardate quali.
   Per ogni modello scrivete nella scheda da quale lezione viene e perché lo avete scelto.
3. Eseguite la gara e leggete la classifica.

Attenzione: alcuni modelli con molti esempi sono lenti (la SVM, per esempio). Se una cella gira per più di un minuto,
fermatela e usate meno esempi o un altro modello.

**Come sapete che funziona.** Il controllo verifica che ci siano almeno 4 modelli con una MAE calcolata.
Se vince la baseline ve lo dice: non è un errore, è un risultato, e va scritto nella scheda.

In [ ]:
def train_model(model, X_train, y_train):
    # addestro il modello sugli esempi di train
    FILL_ME     # scrivete: l'istruzione che addestra il modello
    return model


def evaluate_model(model, X_test, y_test):
    # chiedo al modello la previsione e la confronto con la realtà
    prediction = model.predict(X_test)
    mae = FILL_ME     # scrivete: l'errore assoluto medio tra la realtà e la previsione
    return mae


candidate_models = {
    # --- scrivete voi: almeno 4 modelli da almeno 3 lezioni diverse ---
    # formato:  "nome che volete dare": Modello(parametri),
}

results = []
for model_name, model in candidate_models.items():
    trained_model = train_model(model, X_train, y_train)
    mae = evaluate_model(trained_model, X_test, y_test)
    results.append({"modello": model_name, "MAE": round(mae, 2)})

results.append({"modello": "Baseline (media 3 mesi)", "MAE": round(baseline_mae, 2)})

results_table = pd.DataFrame(results).sort_values("MAE").reset_index(drop=True)
results_table

In [ ]:
# --- Controllo blocco 5 ---
assert len(candidate_models) >= 4, f"Avete {len(candidate_models)} modelli: ne servono almeno 4, da almeno 3 lezioni diverse"
assert results_table["MAE"].notna().all(), "Qualche MAE è vuota: controllate evaluate_model"
best_name = results_table.loc[0, "modello"]
print("OK · blocco 5 · il migliore sul test è:", best_name, "con MAE", results_table.loc[0, "MAE"])
if best_name.startswith("Baseline"):
    print("   La baseline vince: i modelli non aggiungono niente. Succede, ed è un risultato. Scrivetelo nella scheda.")

**Fermatevi un momento, prima di passare al blocco 6.** Nella scheda del gruppo rispondete:

- La classifica vi sorprende? Il modello che vi aspettavate più forte è davvero il più forte?
- Il modello migliore quanto batte la baseline, in pezzi? Se la differenza è di pochi decimi di pezzo, vale la pena di usare
  un modello che nessuno in magazzino sa spiegare?
- Provate a **togliere una delle colonne che avete inventato** (commentatela nel blocco 2, rieseguite i blocchi 2-5): la classifica cambia?
  Questo vi dice se la colonna serviva.

Un avviso: in questo blocco avete già guardato il test. Da adesso in poi ogni volta che lo riguardate per **scegliere** qualcosa,
il test vale un po' meno. Nel blocco 6 le scelte si fanno **senza** il test.

## Blocco 6 · Come scelgo gli iperparametri?

**La domanda.** Un **iperparametro** è un numero che il modello non impara dai dati: lo decidete voi prima di addestrare.
Il `k` del kNN, la profondità di un albero, il numero di alberi di una foresta, il `C` di una SVM. Nel blocco 5 li avete
scritti a mano (`n_neighbors=5`, `max_depth=5`): perché 5 e non 15? Nessun motivo. Questo blocco serve a sostituire
"nessun motivo" con un procedimento.

**L'idea.** Non si scelgono sul test: il test si usa una volta sola, e l'avete già usato. Si scelgono sul train con la
**cross-validation** (lezione 3): il train viene diviso in 5 parti; per 5 volte il modello si addestra su 4 parti e si misura
sulla quinta; la media dei 5 errori è una stima onesta di come andrà su dati nuovi, ottenuta **senza toccare il test**.

`GridSearchCV` fa tutto questo in automatico: gli date un modello e una **griglia** di valori da provare, e lui prova tutte
le combinazioni, per ognuna fa i 5 giri, e tiene la combinazione con l'errore medio più basso.
Chiede una misura "più alta è meglio", quindi la MAE si passa come `neg_mean_absolute_error` (MAE con il segno meno).

**Che cosa dovete fare.**

1. Completate `choose_hyperparameters`: manca la creazione di `GridSearchCV` con 5 parti di cross-validation e la misura giusta.
   La Mappa cerca la documentazione di `GridSearchCV` (i parametri si chiamano `cv` e `scoring`).
2. Riempite `searches`: **almeno 2 dei vostri modelli** del blocco 5, ognuno con una griglia di valori per i suoi iperparametri.
   Nel formato c'è l'esempio del kNN. Per trovare i nomi degli iperparametri di un modello guardate la sua documentazione
   o scrivete `NomeModello().get_params()` in una cella.
   Griglie piccole: 3-6 valori per iperparametro, altrimenti aspettate molto.
3. Eseguite: per ogni modello vedete gli iperparametri scelti e la MAE in cross-validation. **Poi, e solo poi,** la MAE sul test.
4. Confrontate le due MAE (cross-validation e test). Sono vicine? Se sono molto diverse, pensate al perché: i mesi di test
   sono diversi dai mesi di train? Il test è piccolo? Scrivetelo nella scheda.

**Come sapete che funziona.** Il controllo verifica che ci siano almeno 2 ricerche e che ognuna abbia restituito dei parametri.

In [ ]:
def choose_hyperparameters(model, parameter_grid, X_train, y_train):
    # provo tutte le combinazioni della griglia, con 5 parti di cross-validation sul train, misurando la MAE
    search = FILL_ME     # scrivete: la ricerca a griglia con 5 parti e la MAE come misura (con il segno meno)
    search.fit(X_train, y_train)

    best_mae_cv = -search.best_score_
    return search.best_estimator_, search.best_params_, best_mae_cv


searches = {
    # formato:  "nome": (Modello(), {"iperparametro": [valori da provare]}),
    "kNN": (KNeighborsRegressor(), {"n_neighbors": [3, 5, 10, 15, 25, 40]}),
    # --- scrivete voi: almeno un altro modello con la sua griglia ---
}

tuned_models = {}
tuning_rows = []
for search_name, (model, parameter_grid) in searches.items():
    best_model, best_params, mae_cv = choose_hyperparameters(model, parameter_grid, X_train, y_train)
    tuned_models[search_name] = best_model
    tuning_rows.append({"modello": search_name, "iperparametri scelti": best_params, "MAE cross-validation": round(mae_cv, 2)})
    print(search_name, "· scelta in cross-validation:", best_params, f"· MAE cv = {mae_cv:.2f}")

# ora, e solo ora, guardo il test con i modelli scelti
print()
for search_name, best_model in tuned_models.items():
    mae_test = evaluate_model(best_model, X_test, y_test)
    for row in tuning_rows:
        if row["modello"] == search_name:
            row["MAE test"] = round(mae_test, 2)
    print(f"{search_name:20s} · MAE test = {mae_test:.2f}")
print(f"{'Baseline':20s} · MAE test = {baseline_mae:.2f}")

tuning_table = pd.DataFrame(tuning_rows)
tuning_table

In [ ]:
# --- Controllo blocco 6 ---
assert len(searches) >= 2, f"Avete {len(searches)} ricerche: ne servono almeno 2"
for row in tuning_rows:
    assert len(row["iperparametri scelti"]) > 0, f"La ricerca {row['modello']} non ha restituito iperparametri: controllate la griglia"
print("OK · blocco 6 ·", len(searches), "modelli con iperparametri scelti in cross-validation")

### Il modello sigillato

Adesso scegliete **il** modello del gruppo: uno solo. Può essere uno dei modelli con gli iperparametri scelti nel blocco 6
(`tuned_models["kNN"]`, per esempio), uno del blocco 5, o la baseline se nessuno l'ha battuta (in quel caso scrivete `None`
e il notebook userà la baseline anche venerdì).

Motivate la scelta nella scheda: non solo "ha la MAE più bassa", ma anche quanto è semplice da spiegare, quanto è lento,
quanto è stabile tra cross-validation e test.

Da questo momento il modello è **sigillato**: non si cambia più. Venerdì lo provate su settembre così com'è.
È quello che succede in azienda: il modello va in produzione e il mese dopo si scopre com'è andato.

L'ultima cella stampa il **miglioramento rispetto alla baseline in percentuale**: scrivetelo alla lavagna con il nome del gruppo e della famiglia.

In [ ]:
# il modello scelto dal gruppo (cambiate la riga qui sotto)
chosen_model = tuned_models["kNN"]     # <- il vostro modello, oppure None per usare la baseline
chosen_model_name = "kNN con k scelto in cross-validation"     # <- e il suo nome

if chosen_model is None:
    chosen_mae = baseline_mae
    chosen_model_name = "Baseline (media 3 mesi)"
else:
    chosen_model = train_model(chosen_model, X_train, y_train)
    chosen_mae = evaluate_model(chosen_model, X_test, y_test)

improvement_percent = (baseline_mae - chosen_mae) / baseline_mae * 100

print("Modello sigillato:", chosen_model_name)
print(f"MAE test = {chosen_mae:.2f} pezzi · baseline = {baseline_mae:.2f} pezzi")
print(f"Miglioramento rispetto alla baseline: {improvement_percent:+.1f} %   <- alla lavagna")

**Fine della parte di oggi.** Salvate il notebook (`File → Save`). Venerdì ripartite da qui, con un file in più.

---

# Venerdì · Lezione 14

## Settembre arriva

Oggi ricevete `warehouse_september.csv`: per ogni articolo, i pezzi usciti a **settembre 2026**.
È un mese che il vostro modello non ha mai visto, e nemmeno voi: non avete potuto usarlo per scegliere niente.
È il test più onesto che esista, e nella realtà è l'unico che conta: il mese dopo la messa in produzione.

Eseguite la cella: aggiunge settembre allo storico, allunga la lista dei mesi e ricostruisce gli esempi con la **vostra** `make_examples`
(se nel blocco 2 avete rispettato la regola d'oro, funziona senza toccare niente).

Prima di eseguire, scrivete nella scheda la vostra **previsione**: su settembre il modello sigillato andrà meglio, peggio o uguale
rispetto a luglio-agosto? Poi guardate.

In [ ]:
# carico settembre e lo aggiungo allo storico della famiglia
september = pd.read_csv("warehouse_september.csv")
items = items.merge(september, on="item_id", how="left")
data = data.merge(september, on="item_id", how="left")

NEW_MONTH = "2026-09"
MONTHS = MONTHS + [NEW_MONTH]

# ricostruisco gli esempi con il mese in più
example_tables = []
for month in MONTHS[6:]:
    example_tables.append(make_examples(items, month))
examples = pd.concat(example_tables, ignore_index=True)

is_september = examples["target_month"] == NEW_MONTH
X_september = examples[is_september][FEATURES]
y_september = examples[is_september]["target"]

print("Mesi:", MONTHS[0], "->", MONTHS[-1], "(", len(MONTHS), "mesi )")
print("Esempi di settembre:", len(X_september))

# il modello sigillato su settembre, senza toccarlo
if chosen_model is None:
    september_prediction = baseline_predict(X_september)
else:
    september_prediction = chosen_model.predict(X_september)
september_mae = mean_absolute_error(y_september, september_prediction)
september_baseline_mae = mean_absolute_error(y_september, baseline_predict(X_september))
september_improvement = (september_baseline_mae - september_mae) / september_baseline_mae * 100

print()
print(f"Settembre · modello sigillato: MAE = {september_mae:.2f} pezzi")
print(f"Settembre · baseline:          MAE = {september_baseline_mae:.2f} pezzi")
print(f"Miglioramento rispetto alla baseline su settembre: {september_improvement:+.1f} %   <- alla lavagna, accanto a quello di mercoledì")
print(f"(mercoledì, su luglio-agosto, era {improvement_percent:+.1f} %)")

**Nella scheda:** com'è andata rispetto alla vostra previsione? La MAE di settembre è vicina a quella di luglio-agosto e a quella
della cross-validation, o è molto diversa? Se è molto diversa, guardate i dati: settembre è un mese normale per la vostra famiglia?
(La cella del blocco 1 con `random_state` diverso aiuta.) Il miglioramento percentuale ha tenuto?

Questa è la risposta vera a "quanto posso fidarmi del test": il test dice qualcosa sul futuro solo se il futuro assomiglia al passato.

## Blocco 7 · Devo riallenarlo ogni volta?

**La domanda.** Ogni mese arrivano dati nuovi. Il modello va riaddestrato ogni mese? Ogni settimana? Mai?
Chi lo chiede si aspetta una regola. La regola non c'è: si **misura**, come per tutto il resto.

**L'idea.** Confrontiamo due strade, mese per mese, su luglio, agosto e settembre:

- **modello fisso**: addestrato una volta sola con i dati fino a giugno (il vostro modello sigillato è così), e usato per tutti e tre i mesi;
- **modello riallenato**: ogni mese lo riaddestro da zero con **tutti i dati fino al mese prima**. Per luglio ha gli stessi dati del fisso;
  per agosto ha anche luglio; per settembre ha anche luglio e agosto.

Se il riallenato va meglio, i dati freschi contano e riallenare conviene. Se va uguale, riallenare ogni mese è lavoro sprecato
(e ogni riaddestramento è un'occasione per rompere qualcosa). Se va peggio, è un segnale che il mese nuovo era strano.

**Che cosa dovete fare.** Completate `retrain_monthly`. Il modello fisso è già pronto; dentro il ciclo manca il modello riallenato:

1. la condizione che seleziona gli esempi con mese bersaglio **prima** di `month`
   (il confronto tra testi come `"2026-07" < "2026-08"` funziona, perché le date sono scritte anno-mese);
2. una copia nuova del modello (`clone(model)` crea un modello uguale ma non addestrato);
3. il suo addestramento su quegli esempi.

**Come sapete che funziona.** Il controllo verifica che il train del riallenato cresca di un mese alla volta:
a luglio è uguale al train, ad agosto ha un mese in più, a settembre due.

In [ ]:
def retrain_monthly(model, examples, months_to_test):
    rows = []

    # modello fisso: addestrato una volta su tutto quello che c'è prima del primo mese da provare
    is_before_first = examples["target_month"] < months_to_test[0]
    fixed_model = clone(model)
    fixed_model.fit(examples[is_before_first][FEATURES], examples[is_before_first]["target"])

    for month in months_to_test:
        is_month = examples["target_month"] == month
        X_month = examples[is_month][FEATURES]
        y_month = examples[is_month]["target"]

        # --- scrivete voi: il modello riallenato ---
        # 1. la condizione "vero" per gli esempi con mese bersaglio prima di month
        is_before_month = FILL_ME     # scrivete: la condizione
        # 2. una copia nuova del modello, non addestrata
        retrained_model = FILL_ME     # scrivete: la copia
        # 3. l'addestramento della copia sugli esempi selezionati al punto 1
        FILL_ME     # scrivete: l'addestramento

        mae_fixed = mean_absolute_error(y_month, fixed_model.predict(X_month))
        mae_retrained = mean_absolute_error(y_month, retrained_model.predict(X_month))
        mae_baseline = mean_absolute_error(y_month, baseline_predict(X_month))

        rows.append({"mese": month, "esempi nel train riallenato": int(is_before_month.sum()),
                     "MAE fisso": round(mae_fixed, 2), "MAE riallenato": round(mae_retrained, 2), "MAE baseline": round(mae_baseline, 2)})

    return pd.DataFrame(rows)


MONTHS_TO_TEST = ["2026-07", "2026-08", "2026-09"]
model_for_experiment = chosen_model if chosen_model is not None else KNeighborsRegressor(n_neighbors=15)
retrain_table = retrain_monthly(model_for_experiment, examples, MONTHS_TO_TEST)
retrain_table

In [ ]:
# --- Controllo blocco 7 ---
train_sizes = retrain_table["esempi nel train riallenato"].tolist()
assert train_sizes[0] == len(X_train), "A luglio il riallenato deve avere gli stessi esempi del train (fino a giugno)"
assert train_sizes[1] == train_sizes[0] + len(items), "Ad agosto il riallenato deve avere un mese in più (luglio)"
assert train_sizes[2] == train_sizes[0] + 2 * len(items), "A settembre il riallenato deve avere due mesi in più (luglio e agosto)"
print("OK · blocco 7")
print("Differenza media (fisso - riallenato):", round((retrain_table["MAE fisso"] - retrain_table["MAE riallenato"]).mean(), 3), "pezzi")

**Nella scheda:** riallenare ogni mese cambia qualcosa? Di quanto, in pezzi? È una differenza che il responsabile del magazzino
noterebbe? Con 3 mesi di prova si può dare una risposta definitiva? Che cosa servirebbe per darla?

## Blocco 8 · E poi, che cosa ci faccio?

**La domanda.** Una previsione da sola non serve a nessuno. Il responsabile del magazzino non vuole sapere "a ottobre usciranno 4,6 pezzi":
vuole sapere **quanti pezzi tenere** e **quando ordinare**. Questo blocco trasforma la previsione in quei due numeri,
per ogni articolo, e li confronta con quelli del sistema che l'azienda usa davvero.

**L'idea.** Due numeri per articolo:

- **Scorta di sicurezza (SS):** i pezzi in più che tengo per coprire i mesi in cui la domanda è sopra la media.
  Dipende da tre cose: quanto la domanda balla (la deviazione standard), quanto aspetto il fornitore (il lead time: più aspetto,
  più a lungo sono scoperto) e quanto voglio essere sicuro (il coefficiente `k`, che viene dal livello di servizio:
  85% → k = 1,04; 95% → 1,64; 98% → 2,05).
- **Punto di riordino (ROP):** quando la giacenza scende sotto questo numero, faccio l'ordine. Deve coprire la domanda
  **durante** il lead time (perché nel frattempo i pezzi continuano a uscire), più la scorta di sicurezza.

Le formule, nella versione semplice che va bene qui:

```
domanda_giornaliera = previsione_del_mese_prossimo / 30
SS  = k × deviazione_standard_mensile × √(lead_time_days / 30)
ROP = domanda_giornaliera × lead_time_days + SS
```

La radice quadrata serve perché la variabilità cresce con la radice del tempo, non con il tempo: è un risultato di statistica
che potete prendere per buono (chi vuole: cercate "scorta di sicurezza lead time radice").

**Che cosa dovete fare.** La prima cella costruisce gli esempi per **ottobre 2026** (il mese prossimo): ci sono le colonne di `X`,
costruite con la vostra `make_examples` su un mese che non ha ancora il bersaglio, e la previsione del modello sigillato.

Poi scrivete voi il corpo di `decide_stock`, seguendo i commenti numerati: domanda giornaliera, deviazione standard mensile
(usate gli ultimi 6 mesi di storico: `items[MONTHS[-6:]].std(axis=1)`), scorta di sicurezza, punto di riordino.
`items["k"]` e `items["lead_time_days"]` sono le colonne che servono. Arrotondate a numeri interi: in magazzino i pezzi sono interi.

Infine la cella di confronto carica `warehouse_policy_reference.csv`: per ogni articolo, la scorta di sicurezza e il livello minimo
di giacenza calcolati dal sistema aziendale (`safety_stock_system`, `min_stock_system`). Il livello minimo del sistema è
il suo punto di riordino.

**Come sapete che funziona.** Il controllo rifà la formula a mano sul primo articolo e verifica che nessun punto di riordino
sia sotto la sua scorta di sicurezza.

In [ ]:
# esempi per il mese prossimo (ottobre 2026): aggiungo una colonna vuota e uso la vostra make_examples
NEXT_MONTH = "2026-10"
items_with_next = items.copy()
items_with_next[NEXT_MONTH] = 0          # il bersaglio non esiste ancora: lo metto a zero e lo ignoro
MONTHS = MONTHS + [NEXT_MONTH]
next_month = make_examples(items_with_next, NEXT_MONTH).reset_index(drop=True)
MONTHS = MONTHS[:-1]                     # tolgo ottobre dalla lista dei mesi veri

# previsione per ottobre con il modello sigillato (una previsione negativa non ha senso: la porto a zero)
if chosen_model is None:
    forecast = baseline_predict(next_month[FEATURES])
else:
    forecast = chosen_model.predict(next_month[FEATURES])
forecast = np.clip(forecast, 0, None)
next_month["forecast_next_month"] = np.round(forecast, 1)

print("Previsione per", NEXT_MONTH, "pronta per", len(next_month), "articoli")
next_month[["item_id", "forecast_next_month"]].head()

In [ ]:
def decide_stock(next_month, items):
    decision = pd.DataFrame()
    decision["item_id"] = items["item_id"]
    decision["forecast_next_month"] = next_month["forecast_next_month"]

    # --- scrivete voi ---
    # 1. domanda giornaliera: previsione del mese prossimo diviso 30

    # 2. deviazione standard mensile: sugli ultimi 6 mesi di storico, riga per riga

    # 3. scorta di sicurezza: k x deviazione standard x radice(lead time / 30)

    # 4. punto di riordino: domanda giornaliera x lead time + scorta di sicurezza

    # 5. mettete i due numeri, arrotondati a interi, nelle colonne "safety_stock" e "reorder_point" di decision

    return decision


decision = decide_stock(next_month, items)
decision.head(10)

In [ ]:
# --- Controllo blocco 8 ---
assert "safety_stock" in decision.columns and "reorder_point" in decision.columns, "Mancano le colonne safety_stock e reorder_point"
first_daily = next_month.loc[0, "forecast_next_month"] / 30
first_std = items.loc[0, MONTHS[-6:]].std()
expected_ss = items.loc[0, "k"] * first_std * np.sqrt(items.loc[0, "lead_time_days"] / 30)
expected_rop = first_daily * items.loc[0, "lead_time_days"] + expected_ss
assert abs(decision.loc[0, "safety_stock"] - round(expected_ss, 0)) <= 1, "Scorta di sicurezza: deve essere k x deviazione standard mensile x radice(lead time / 30)"
assert abs(decision.loc[0, "reorder_point"] - round(expected_rop, 0)) <= 1, "Punto di riordino: deve essere domanda giornaliera x lead time + scorta di sicurezza"
assert (decision["reorder_point"] >= decision["safety_stock"] - 1).all(), "Il punto di riordino non può essere sotto la scorta di sicurezza"
print("OK · blocco 8")

In [ ]:
# confronto con il sistema dell'azienda
reference = pd.read_csv("warehouse_policy_reference.csv")
comparison = decision.merge(reference, on="item_id")

# tengo solo gli articoli per cui il sistema ha calcolato una politica di scorta (livello minimo > 0)
has_policy = comparison["min_stock_system"] > 0
comparison = comparison[has_policy].copy()

comparison["ss_difference"] = comparison["safety_stock"] - comparison["safety_stock_system"]
comparison["rop_difference"] = comparison["reorder_point"] - comparison["min_stock_system"]

print("Articoli con una politica di scorta nel sistema:", len(comparison))
print(f"Scorta di sicurezza · media nostra {comparison['safety_stock'].mean():.1f} · media sistema {comparison['safety_stock_system'].mean():.1f}")
print(f"Punto di riordino   · media nostra {comparison['reorder_point'].mean():.1f} · media sistema {comparison['min_stock_system'].mean():.1f}")
print(f"Differenza assoluta media sul punto di riordino: {comparison['rop_difference'].abs().mean():.1f} pezzi")
total_value_ours = (comparison["reorder_point"] * items.set_index("item_id").loc[comparison["item_id"], "unit_cost"].values).sum()
total_value_system = (comparison["min_stock_system"] * items.set_index("item_id").loc[comparison["item_id"], "unit_cost"].values).sum()
print(f"Valore della merce al punto di riordino · nostro {total_value_ours:,.0f} € · sistema {total_value_system:,.0f} €")

figure, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(comparison["safety_stock_system"], comparison["safety_stock"], alpha=0.5)
axes[0].plot([0, comparison["safety_stock_system"].max()], [0, comparison["safety_stock_system"].max()], color="gray")
axes[0].set_xlabel("scorta di sicurezza · sistema")
axes[0].set_ylabel("scorta di sicurezza · nostra")
axes[1].scatter(comparison["min_stock_system"], comparison["reorder_point"], alpha=0.5)
axes[1].plot([0, comparison["min_stock_system"].max()], [0, comparison["min_stock_system"].max()], color="gray")
axes[1].set_xlabel("livello minimo · sistema")
axes[1].set_ylabel("punto di riordino · nostro")
plt.tight_layout()
plt.show()

# i 10 articoli su cui siamo più lontani dal sistema
comparison.reindex(comparison["rop_difference"].abs().sort_values(ascending=False).index).head(10)

**Nella scheda:** siete sopra o sotto il sistema dell'azienda? Su quali articoli siete più lontani, e perché (guardate il loro storico)?
Stare sopra costa magazzino (soldi fermi sugli scaffali), stare sotto costa clienti (pezzi che mancano): quale errore preferite,
e per quali articoli (A o C)?

Una cosa da tenere a mente: il sistema dell'azienda usa più informazioni di voi (lotti minimi dei fornitori, politiche diverse per articolo,
giacenza attuale). La domanda non è "chi ha ragione", ma "la vostra catena di montaggio dà numeri sensati?". Se per un articolo
proponete 400 pezzi di scorta e il sistema ne tiene 50, uno dei due ha capito qualcosa che l'altro non ha capito: trovate che cosa.

---

## Scheda del gruppo (la compila il Diario, la presenta la Voce)

Scrivete qui, in questa cella (doppio clic per modificarla). Compilatela **man mano**, non alla fine.

**Gruppo:** ___ · **Componenti e ruoli:** ___

**Famiglia scelta:** ___ · **Articoli dopo il filtro:** ___

**Blocco 1 · Come sono i dati?** (regolari, a ondate, un solo picco; quale tipo sarà più difficile e perché)
___

**Blocco 2 · Colonne inventate** (nome, che cosa misura, perché dovrebbe aiutare):
1. ___
2. ___
3. ___

**Blocco 4 · Baseline, MAE sul test (luglio-agosto):** ___ pezzi

**Blocco 5 · Modelli in gara** (nome, lezione da cui viene, MAE sul test, dal migliore):
1. ___
2. ___
3. ___
4. ___

La classifica vi ha sorpreso? Togliere una colonna ha cambiato qualcosa? ___

**Blocco 6 · Iperparametri scelti in cross-validation** (modello, valori, MAE cv, MAE test):
___

MAE in cross-validation e MAE sul test sono vicine? Se no, perché secondo voi? ___

**Modello sigillato e perché:** ___ · **Miglioramento rispetto alla baseline:** ___ %

**Venerdì · Previsione prima di vedere settembre** (meglio / peggio / uguale): ___
**Settembre, com'è andata** (MAE modello, MAE baseline, miglioramento %): ___
Perché, secondo voi? ___

**Blocco 7 · Riallenare ogni mese cambia qualcosa?** ___

**Blocco 8 · Rispetto al sistema dell'azienda siamo** (sopra / sotto / vicini): ___
Articoli più lontani e perché: ___

**Una cosa che abbiamo capito che prima non ci era chiara:** ___

**Una cosa che ancora non ci torna:** ___

## Extra (per chi finisce prima)

- **E1.** La MAE media nasconde le differenze tra articoli: calcolate la MAE separatamente per classe A, B e C
  (`test_examples` ha `item_id`, `items` ha `abc_class`: unitele). Dove sbaglia di più il modello? È un problema?
- **E2.** Nel blocco 6 la cross-validation mescola i mesi, cioè fa proprio quello che nel blocco 3 abbiamo detto di non fare.
  Provate `TimeSeriesSplit` al posto di `cv=5` (documentazione di scikit-learn): è il modo onesto di fare cross-validation nel tempo.
  Cambia la scelta degli iperparametri? La MAE in cross-validation si avvicina a quella del test?
- **E3.** Nel blocco 8 provate un livello di servizio diverso per tutti gli articoli (per esempio `k = 2.05`, cioè 98%).
  Di quanto cresce la scorta totale, in pezzi e in euro (`unit_cost`)? Vale la pena?
- **E4.** Il modello prevede un numero di pezzi. Trasformatelo in un classificatore: "il mese prossimo esce almeno 1 pezzo, sì o no?"
  e misuratelo con la matrice di confusione (lezione 3). Per quali articoli è più utile questa domanda invece della quantità?